<a href="https://colab.research.google.com/github/Noors-lab/VigIQ-Vigilance-Intelligence-/blob/main/interface_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import zipfile
import os
from google.colab import drive

drive.mount('/content/drive')

root = "/content/drive/MyDrive/shoplifting"
extract_to = "/content/shoplifting_data"
os.makedirs(extract_to, exist_ok=True)

for file in os.listdir(root):
    if file.endswith('.zip'):
        zip_path = os.path.join(root, file)
        folder_name = file.replace('.zip', '').replace('.', '_')
        out_path = os.path.join(extract_to, folder_name)
        if not os.path.exists(out_path):
            print(f"Extracting {file}...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(out_path)
            print(f"  Done → {out_path}")
        else:
            print(f"Already extracted: {file}")

print("\nAll ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting shoplifting.zip...
  Done → /content/shoplifting_data/shoplifting
Extracting shoplifting_1.zip...
  Done → /content/shoplifting_data/shoplifting_1
Extracting shoplifting.v1i.yolov11.zip...
  Done → /content/shoplifting_data/shoplifting_v1i_yolov11
Extracting Shoplifting.v2i.yolov11.zip...
  Done → /content/shoplifting_data/Shoplifting_v2i_yolov11

All ready.


In [5]:
import os
path = "/content/shoplifting_data/shoplifting/shoplifting/shoplifting-13.mp4"
print("Exists:", os.path.exists(path))

Exists: True


In [6]:
!pip install ultralytics
import cv2
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict
from ultralytics import YOLO
from google.colab import drive
import os
import json
import time
from datetime import datetime

drive.mount('/content/drive')

# ─── LSTM Model ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256,
                 num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.classifier(out).squeeze()

# ─── Load everything ───
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

lstm_model = ShopliftingLSTM().to(device)
lstm_model.load_state_dict(torch.load(
    '/content/drive/MyDrive/shoplifting/vigiq_best_v4.pth',
    map_location=device
))
lstm_model.eval()

X_mean = np.load('/content/drive/MyDrive/shoplifting/X_mean_v4.npy')[0]
X_std  = np.load('/content/drive/MyDrive/shoplifting/X_std_v4.npy')[0]

yolo = YOLO("yolo11n-pose.pt")

print(f"Models loaded on {device} ✓")

# ─── Keypoint indices ───
NOSE = 0
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_WRIST,    RIGHT_WRIST    = 9, 10
LEFT_HIP,      RIGHT_HIP      = 11, 12

# ─── Normalize keypoints ───
def normalize_keypoints(kps):
    left_hip   = kps[LEFT_HIP]
    right_hip  = kps[RIGHT_HIP]
    center     = (left_hip + right_hip) / 2
    l_shoulder = kps[LEFT_SHOULDER]
    r_shoulder = kps[RIGHT_SHOULDER]
    shoulder_c = (l_shoulder + r_shoulder) / 2
    scale      = np.linalg.norm(shoulder_c - center)
    if scale < 1e-6:
        scale = 1.0
    return ((kps - center) / scale).flatten().tolist()

# ─── Rule layer ───
def apply_rules(kps_sequence):
    if len(kps_sequence) < 10:
        return False, "insufficient frames", 0.0

    seq = np.array(kps_sequence[-50:], dtype=np.float32)
    while len(seq) < 50:
        seq = np.vstack([seq, np.zeros((1, 34))])
    seq = (seq - X_mean) / (X_std + 1e-8)
    tensor = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logit = lstm_model(tensor)
        confidence = torch.sigmoid(logit).item()

    if confidence < 0.80:
        return False, f"low confidence ({confidence:.2f})", confidence

    recent = np.array(kps_sequence[-15:])
    if len(recent) >= 5:
        lw_x = recent[:, LEFT_WRIST*2]
        rw_x = recent[:, RIGHT_WRIST*2]
        wrist_check = (abs(lw_x[-1] - lw_x[0]) > 0.1 or
                       abs(rw_x[-1] - rw_x[0]) > 0.1)
    else:
        wrist_check = False

    if not wrist_check:
        return False, "wrist not moving", confidence

    if len(kps_sequence) < 20:
        return False, "not sustained", confidence

    nose_x    = np.array(kps_sequence[-20:])[:, NOSE*2]
    head_rotating = np.var(nose_x) > 0.01

    nose_pos  = np.array(kps_sequence[-10:])[:, NOSE*2]
    hooded    = np.mean(np.abs(nose_pos)) < 0.05

    if hooded and confidence >= 0.65:
        return True, "HOODED PERSON + suspicious motion", confidence

    if head_rotating:
        return True, "suspicious motion + head rotation", confidence

    return True, "suspicious motion detected", confidence

# ─── Clip extractor ───
def extract_clip(frame_buffer, fps, output_path):
    if not frame_buffer:
        return
    h, w = frame_buffer[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    for frame in frame_buffer:
        out.write(frame)
    out.release()

# ─── Alert logger ───
def log_alert(alert_log, person_id, reason, confidence, timestamp, clip_path):
    alert = {
        "person_id":  int(person_id),
        "reason":     reason,
        "confidence": round(float(confidence), 2),
        "timestamp":  timestamp,
        "clip":       clip_path,
        "time":       datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    alert_log.append(alert)
    print(f"\n🚨 ALERT")
    print(f"   Person:     {person_id}")
    print(f"   Reason:     {reason}")
    print(f"   Confidence: {confidence:.2f}")
    print(f"   Timestamp:  {timestamp:.1f}s")
    print(f"   Clip:       {clip_path}")
    return alert

# ─── Main pipeline ───
def run_pipeline(source, output_dir="/content/vigiq_alerts", max_frames=None):
    """
    source: path to video file OR rtsp:// URL
    output_dir: where to save alert clips
    max_frames: limit frames for testing (None = run fully)
    """
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(source)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    # Rolling frame buffer — keeps last 10 seconds
    BUFFER_SIZE = int(fps * 10)
    frame_buffer = []

    kps_sequences = defaultdict(list)
    alerted_ids   = set()
    alert_log     = []

    frame_idx  = 0
    SAMPLE_EVERY = 3

    print(f"\n{'='*50}")
    print(f"VigIQ Pipeline Started")
    print(f"Source: {source}")
    print(f"FPS: {fps:.1f}")
    print(f"{'='*50}\n")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if max_frames and frame_idx >= max_frames:
            break

        # Keep rolling frame buffer
        frame_buffer.append(frame.copy())
        if len(frame_buffer) > BUFFER_SIZE:
            frame_buffer.pop(0)

        # Process every 3rd frame
        if frame_idx % SAMPLE_EVERY == 0:
            results = yolo.track(frame, persist=True, verbose=False)

            if (results[0].boxes.id is not None and
                    results[0].keypoints is not None):

                track_ids = results[0].boxes.id.cpu().numpy().astype(int)
                keypoints = results[0].keypoints.xy.cpu().numpy()

                for tid, kps in zip(track_ids, keypoints):
                    if kps.shape[0] != 17:
                        continue

                    # Add to trajectory
                    normalized = normalize_keypoints(kps)
                    kps_sequences[tid].append(normalized)

                    if len(kps_sequences[tid]) > 50:
                        kps_sequences[tid].pop(0)

                    # Skip already alerted
                    if tid in alerted_ids:
                        continue

                    # Apply rules every 10 frames
                    if frame_idx % 10 != 0:
                        continue

                    alert, reason, conf = apply_rules(kps_sequences[tid])

                    if alert:
                        timestamp  = frame_idx / fps
                        clip_name  = f"alert_{tid}_{int(timestamp)}s.mp4"
                        clip_path  = os.path.join(output_dir, clip_name)

                        # Save clip
                        extract_clip(frame_buffer, fps, clip_path)

                        # Log alert
                        log_alert(alert_log, tid, reason,
                                  conf, timestamp, clip_path)

                        alerted_ids.add(tid)

        frame_idx += 1

    cap.release()

    # Save alert log
    log_path = os.path.join(output_dir, "alert_log.json")
    with open(log_path, 'w') as f:
        json.dump(alert_log, f, indent=2)

    print(f"\n{'='*50}")
    print(f"Pipeline finished.")
    print(f"Total frames:  {frame_idx}")
    print(f"Total alerts:  {len(alert_log)}")
    print(f"Log saved:     {log_path}")
    print(f"{'='*50}")

    return alert_log

# ─── Test on shoplifting clip ───
alerts = run_pipeline(
    source="/content/shoplifting_data/shoplifting/shoplifting/shoplifting-13.mp4",
    output_dir="/content/vigiq_alerts",
    max_frames=500
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Models loaded on cuda ✓

VigIQ Pipeline Started
Source: /content/shoplifting_data/shoplifting/shoplifting/shoplifting-13.mp4
FPS: 30.0

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 408ms
Prepared 1 package in 29ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


🚨 ALERT
   Person:     1
   Reason:     suspicious motion detected
   Confidence: 0.88
   Timestamp:  7.0s
   Clip:       /content/vigiq_alerts/alert_1_7s.mp4

Pipeline finished.
Total frames:  334
Total alerts:  1
Log saved:     /content/vigiq_alerts/alert_log.json


In [7]:
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Save alert clips to Drive
drive_alerts = "/content/drive/MyDrive/shoplifting/vigiq_alerts"
if os.path.exists(drive_alerts):
    shutil.rmtree(drive_alerts)
shutil.copytree("/content/vigiq_alerts", drive_alerts)
print(f"Alerts saved to Drive ✓")
print(f"Files: {os.listdir(drive_alerts)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Alerts saved to Drive ✓
Files: ['alert_log.json', 'alert_1_7s.mp4']
